# Aula 7 - Otimização de hiperparâmetros
## Grid search (slides 9–12)


In [1]:
import numpy as np
from si.data.dataset import Dataset
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy
from si.metrics.accuracy import accuracy
from si.model_selection.cross_validation import k_fold_cross_validation
from si.model_selection.grid_search import grid_search_cv

In [2]:
def build_network(learning_rate, n_hidden):
    """
    Função construtora (slide 10): devolve uma rede nova, por treinar, com estes hiperparâmetros.
    O grid search chama-a uma vez por fold, por isso nunca se reaproveitam pesos.
    """
    # a mesma seed em todas as chamadas: as redes começam com os mesmos pesos iniciais e usam os
    # mesmos mini-batches, por isso as diferenças nos scores vêm só dos hiperparâmetros
    np.random.seed(0)
    net = NeuralNetwork(epochs=100, batch_size=16, optimizer=SGD,
                        learning_rate=learning_rate, momentum=0.9,
                        loss=BinaryCrossEntropy, metric=accuracy)
    # 2 features --> n_hidden neurónios (ReLU) --> 1 saída (sigmoide: probabilidade de estar dentro do círculo)
    net.add(DenseLayer(n_hidden, (2,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net

In [3]:
# dados (slide 12): 400 pontos no quadrado [-2, 2] x [-2, 2]
# classe 1 se o ponto estiver dentro do círculo x1² + x2² < 1.5
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, size=(400, 2))
y = (X[:, 0]**2 + X[:, 1]**2 < 1.5).astype(float)
ds = Dataset(X, y)
print("proporção da classe 0 (fora do círculo):", round(float(1 - y.mean()), 3))

# 3 learning rates x 3 tamanhos da camada escondida = 9 configurações, cada uma treinada em 5 folds
grid = {'learning_rate': [0.001, 0.01, 0.1], 'n_hidden': [2, 4, 16]}
results = grid_search_cv(build_network, ds, grid, cv=5, seed=1)
for h, s in zip(results['hyperparameters'], results['scores']):
    print(h, round(s, 3))
print("melhor:", results['best_hyperparameters'], round(results['best_score'], 3))
print("modelos treinados:", len(results['hyperparameters']) * 5)

# as configurações com 0.715: a rede prevê sempre a mesma classe?
for lr, n in [(0.001, 2), (0.001, 4)]:
    net = build_network(lr, n)
    net.fit(ds)
    print(f"lr {lr}, {n} neurónios -> previsões diferentes: {np.unique(np.round(net.predict(ds)))}")

proporção da classe 0 (fora do círculo): 0.715
{'learning_rate': 0.001, 'n_hidden': 2} 0.715
{'learning_rate': 0.001, 'n_hidden': 4} 0.715
{'learning_rate': 0.001, 'n_hidden': 16} 0.843
{'learning_rate': 0.01, 'n_hidden': 2} 0.725
{'learning_rate': 0.01, 'n_hidden': 4} 0.758
{'learning_rate': 0.01, 'n_hidden': 16} 0.975
{'learning_rate': 0.1, 'n_hidden': 2} 0.725
{'learning_rate': 0.1, 'n_hidden': 4} 0.75
{'learning_rate': 0.1, 'n_hidden': 16} 0.97
melhor: {'learning_rate': 0.01, 'n_hidden': 16} 0.975
modelos treinados: 45
lr 0.001, 2 neurónios -> previsões diferentes: [0.]
lr 0.001, 4 neurónios -> previsões diferentes: [0.]


In [4]:
# última pergunta do slide 12: passar uma rede já criada, em vez da função construtora
try:
    grid_search_cv(build_network(0.01, 16), ds, {'learning_rate': [0.0001, 0.1]}, cv=5, seed=1)
except AttributeError as erro:
    print("AttributeError:", erro)

# sem a verificação do hasattr, o setattr criava o atributo net.learning_rate, que ninguém lê:
# o optimizer foi criado no __init__ com 0.01 e copiado para as camadas no add
for lr in [0.0001, 0.1]:
    net = build_network(0.01, 16)
    net.learning_rate = lr                                   # o que o setattr faria
    scores = k_fold_cross_validation(net, ds, cv=5, seed=1)  # os mesmos folds do grid search
    print(f"net.learning_rate = {lr} | learning_rate do optimizer da 1.ª camada: "
          f"{net.layers[0].w_opt.learning_rate} | score: {np.mean(scores):.3f}")

# com a função construtora, o learning_rate muda mesmo
r = grid_search_cv(build_network, ds, {'learning_rate': [0.0001, 0.1], 'n_hidden': [16]}, cv=5, seed=1)
print("com a função construtora:", [round(s, 3) for s in r['scores']])

AttributeError: O modelo NeuralNetwork não tem o hiperparâmetro 'learning_rate'.
net.learning_rate = 0.0001 | learning_rate do optimizer da 1.ª camada: 0.01 | score: 0.975
net.learning_rate = 0.1 | learning_rate do optimizer da 1.ª camada: 0.01 | score: 0.975
com a função construtora: [0.715, 0.97]


**Conclusão (grid search, slide 12):**
- Melhor configuração: learning_rate 0.01 com 16 neurónios, score 0.975 (logo a seguir, lr 0.1 com 16: 0.970). O número de neurónios pesa mais do que o learning_rate: com 2 ou 4 neurónios, nenhuma configuração passa de 0.76. Com ReLU, cada neurónio escondido acrescenta uma reta à fronteira de decisão, e para aproximar um círculo são precisas muitas retas.
- Várias configurações dão 0.715 porque 0.715 é a proporção da classe 0 (pontos fora do círculo). Com lr 0.001 e 2 ou 4 neurónios, em 100 épocas a rede não aprende e prevê sempre a classe 0 (as previsões são todas 0). É o score de um modelo que não aprendeu nada: serve de referência mínima.
- Modelos treinados: 9 configurações × 5 folds = 45. O custo é configurações × k, e com grids maiores cresce depressa.
- Passar `build_network(0.01, 16)` com `{'learning_rate': [0.0001, 0.1]}`: na minha `grid_search_cv` dá `AttributeError`, porque a `NeuralNetwork` não tem o atributo `learning_rate` (só o optimizer o tem). Sem essa verificação, o `setattr` criava um atributo que ninguém lê, e os dois scores ficavam 0.975, o score de lr 0.01. Com a função construtora, os scores mudam mesmo (0.715 e 0.970). É por isso que a rede precisa de uma função construtora.

## Exercício 17 - randomized_search_cv

In [5]:
from si.model_selection.randomized_search import randomized_search_cv, log_uniform

# log_uniform: cada ordem de grandeza tem a mesma probabilidade (slide 7)
rng = np.random.default_rng(0)
amostrador = log_uniform(1e-4, 1e-1)
log_valores = np.array([amostrador(rng) for _ in range(10000)])
uni_valores = rng.uniform(1e-4, 1e-1, size=10000)
for nome, v in [("log_uniform", log_valores), ("uniform    ", uni_valores)]:
    fracoes = [float(np.mean((v >= 10.0**a) & (v < 10.0**(a + 1)))) for a in (-4, -3, -2)]
    print(f"{nome}: fração em [1e-4, 1e-3), [1e-3, 1e-2), [1e-2, 1e-1): {[round(f, 3) for f in fracoes]}")

# randomized search com o mesmo orçamento do grid search de cima: 9 configurações x 5 folds
distribuicoes = {'learning_rate': log_uniform(1e-3, 1e-1), 'n_hidden': [2, 4, 16]}
results_rand = randomized_search_cv(build_network, ds, distribuicoes, cv=5, n_iter=9, seed=1)
for h, s in zip(results_rand['hyperparameters'], results_rand['scores']):
    print({'learning_rate': round(h['learning_rate'], 4), 'n_hidden': h['n_hidden']}, round(s, 3))
print("melhor:", results_rand['best_hyperparameters'], round(results_rand['best_score'], 3))
print("learning rates diferentes experimentados:",
      len({h['learning_rate'] for h in results_rand['hyperparameters']}), "(no grid search: 3)")

# a mesma seed dá as mesmas configurações (resultados reprodutíveis)
repetido = randomized_search_cv(build_network, ds, distribuicoes, cv=5, n_iter=9, seed=1)
print("mesma seed, mesmas configurações:", repetido['hyperparameters'] == results_rand['hyperparameters'])

log_uniform: fração em [1e-4, 1e-3), [1e-3, 1e-2), [1e-2, 1e-1): [0.334, 0.331, 0.335]
uniform    : fração em [1e-4, 1e-3), [1e-3, 1e-2), [1e-2, 1e-1): [0.01, 0.089, 0.901]
{'learning_rate': 0.0106, 'n_hidden': 16} 0.975
{'learning_rate': 0.0019, 'n_hidden': 16} 0.92
{'learning_rate': 0.0789, 'n_hidden': 2} 0.698
{'learning_rate': 0.007, 'n_hidden': 2} 0.682
{'learning_rate': 0.0452, 'n_hidden': 2} 0.728
{'learning_rate': 0.0126, 'n_hidden': 4} 0.757
{'learning_rate': 0.0011, 'n_hidden': 16} 0.865
{'learning_rate': 0.0119, 'n_hidden': 16} 0.975
{'learning_rate': 0.0046, 'n_hidden': 4} 0.738
melhor: {'learning_rate': 0.010559497443981639, 'n_hidden': 16} 0.975
learning rates diferentes experimentados: 9 (no grid search: 3)
mesma seed, mesmas configurações: True


**Conclusão (Exercício 17):**
- `log_uniform(1e-4, 1e-1)`: ~1/3 dos valores em cada ordem de grandeza. Com uma uniforme normal, 90% caem entre 0.01 e 0.1 e só 1% entre 1e-4 e 1e-3: os learning rates pequenos quase nunca eram experimentados.
- Com o mesmo orçamento do grid search (9 configurações × 5 folds = 45 treinos), o randomized search experimentou 9 learning rates diferentes, contra 3, e chegou ao mesmo melhor score: 0.975 (lr ≈ 0.011, 16 neurónios). Quando só alguns hiperparâmetros importam, explora melhor com o mesmo custo (slide 6).
- Os sorteios usam um gerador próprio (`np.random.default_rng(seed)`), independente do `np.random.seed(0)` da `build_network`: a mesma seed dá as mesmas configurações e os mesmos scores.
- O `_make_folds` e o `_cross_validate` são os do `grid_search.py`: os dois métodos só diferem na forma de escolher as configurações.

## Exercício 18 - afinar a rede do Exercício 16

In [6]:
from si.model_selection.split import train_test_split

# dados do Exercício 16: 500 amostras, 32 features, classificação binária
np.random.seed(42)
X16 = np.random.randn(500, 32)
pesos_escondidos = np.random.randn(32)
# o np.errstate esconde avisos falsos do numpy no Mac (biblioteca Accelerate); os valores estão certos
with np.errstate(all='ignore'):
    y16 = (X16 @ pesos_escondidos + 0.5 * np.random.randn(500) > 0).astype(float)
dataset16 = Dataset(X16, y16)

# o teste fica de fora até ao fim: a pesquisa só vê os dados de treino
train, test = train_test_split(dataset16, test_size=0.2)
print("treino:", train.shape(), "| teste:", test.shape())


def build_network_ex18(learning_rate, batch_size, epochs):
    """
    Função construtora com a arquitetura do Exercício 16 (32 -> 16 -> 8 -> 1).
    Os hiperparâmetros a afinar são o learning_rate, o batch_size e as epochs.
    """
    # os mesmos pesos iniciais e os mesmos mini-batches em todas as chamadas
    np.random.seed(0)
    net = NeuralNetwork(epochs=epochs, batch_size=batch_size, optimizer=SGD, learning_rate=learning_rate,
                        loss=BinaryCrossEntropy, metric=accuracy)
    net.add(DenseLayer(16, (32,)))
    net.add(ReLUActivation())
    net.add(DenseLayer(8))
    net.add(ReLUActivation())
    net.add(DenseLayer(1))
    net.add(SigmoidActivation())
    return net


# 20 configurações sorteadas x 3 folds = 60 treinos
distribuicoes18 = {'learning_rate': log_uniform(1e-4, 1e-1),
                   'batch_size': [8, 16, 32, 64],
                   'epochs': [50, 100, 200]}
results18 = randomized_search_cv(build_network_ex18, train, distribuicoes18, scoring=accuracy,
                                 cv=3, n_iter=20, seed=42)

# ordenar da melhor para a pior configuração
for h, s in sorted(zip(results18['hyperparameters'], results18['scores']), key=lambda par: -par[1]):
    print({'learning_rate': round(h['learning_rate'], 5), 'batch_size': h['batch_size'], 'epochs': h['epochs']},
          round(s, 3))
print("melhor:", results18['best_hyperparameters'], round(results18['best_score'], 3))

treino: (400, 32) | teste: (100, 32)
{'learning_rate': 0.08168, 'batch_size': 16, 'epochs': 200} 0.935
{'learning_rate': 0.03765, 'batch_size': 8, 'epochs': 200} 0.932
{'learning_rate': 0.0188, 'batch_size': 32, 'epochs': 100} 0.92
{'learning_rate': 0.00854, 'batch_size': 16, 'epochs': 200} 0.92
{'learning_rate': 0.02163, 'batch_size': 64, 'epochs': 50} 0.917
{'learning_rate': 0.0013, 'batch_size': 8, 'epochs': 200} 0.917
{'learning_rate': 0.02098, 'batch_size': 32, 'epochs': 100} 0.917
{'learning_rate': 0.00461, 'batch_size': 64, 'epochs': 50} 0.915
{'learning_rate': 0.0304, 'batch_size': 16, 'epochs': 100} 0.915
{'learning_rate': 0.01921, 'batch_size': 32, 'epochs': 200} 0.912
{'learning_rate': 0.01715, 'batch_size': 16, 'epochs': 200} 0.91
{'learning_rate': 0.00251, 'batch_size': 16, 'epochs': 50} 0.905
{'learning_rate': 0.00256, 'batch_size': 64, 'epochs': 50} 0.905
{'learning_rate': 0.00214, 'batch_size': 16, 'epochs': 50} 0.9
{'learning_rate': 0.00048, 'batch_size': 32, 'epochs':

In [7]:
# configuração do Exercício 16, avaliada nos mesmos 3 folds (a mesma seed): comparação justa
config_ex16 = {'learning_rate': 0.01, 'batch_size': 16, 'epochs': 100}
cv_ex16 = grid_search_cv(build_network_ex18, train, {k: [v] for k, v in config_ex16.items()},
                         scoring=accuracy, cv=3, seed=42)['best_score']

# refit: as duas configurações treinadas com o treino todo e avaliadas uma única vez no teste
for nome, config, score_cv in [("afinada     ", results18['best_hyperparameters'], results18['best_score']),
                               ("Exercício 16", config_ex16, cv_ex16)]:
    net = build_network_ex18(**config)
    net.fit(train)
    print(f"{nome}: CV {score_cv:.3f} | treino {net.score(train):.3f} | teste {net.score(test):.3f}")

afinada     : CV 0.935 | treino 1.000 | teste 0.890
Exercício 16: CV 0.922 | treino 1.000 | teste 0.890


**Conclusão (Exercício 18):**
- Melhor configuração: learning_rate ≈ 0.082, batch_size 16, 200 épocas, com accuracy 0.935 na CV. Os learning rates abaixo de ~0.001 são os piores (0.68–0.885): com passos tão pequenos, a rede não chega a aprender nas épocas que tem. A partir de ~0.002, quase todas as configurações ficam entre 0.90 e 0.935: há um patamar largo de configurações boas.
- Na CV, a configuração afinada fica à frente da do Ex. 16 (0.935 contra 0.922, nos mesmos 3 folds), uma diferença de ~5 amostras em 400.
- No teste, as duas dão 0.890: o tuning não melhorou o resultado em dados novos. A configuração do Ex. 16 já estava no patamar bom. Além disso, o melhor score da CV é otimista: escolher o máximo de 20 estimativas com ruído favorece a configuração que teve sorte nos folds. Com 100 amostras de teste, cada amostra vale 0.01 e o desvio padrão da accuracy é ~0.03, por isso diferenças de 0.01–0.02 não se distinguem do ruído.
- As duas dão 1.000 no treino: há overfitting em ambas, e o learning_rate, o batch_size e as epochs não o resolvem. Para isso, a pesquisa teria de incluir hiperparâmetros como o Dropout ou o tamanho da rede.
- O 0.922 não é igual ao 0.908 do Ex. 16 porque aqui a CV tem 3 folds, só usa os 400 de treino, e os pesos iniciais vêm do `np.random.seed(0)` da função construtora.
- O teste só foi usado uma vez, no fim, depois de escolhida a configuração.